### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "SMOTE" / "base_rf_SMOTE_midthres_2026-05-06" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.89,,,,,
1,0,cf_1,,,,,,,17.4055,,,0.9484,1,False,0.1667,0.1567
2,0,cf_2,,,,7,,,18.9745,,,0.8756,2,False,0.1667,0.27
3,0,cf_3,3,,,,,,18.9386,,,0.8772,2,False,0.1667,0.1967
4,0,cf_4,,,,,,,18.0396,5,,0.919,2,False,0.1667,0.1933
5,0,cf_5,,,6,,,,17.6478,,,0.9371,2,False,0.1667,0.14
6,0,cf_6,,,,7,,,16.6455,,,0.9837,2,False,0.1667,0.3
7,0,cf_7,,,,6,,,16.2935,,,1.0,2,False,0.1667,0.36
8,0,cf_8,,,6,7,,,18.9745,,,0.8756,3,False,0.1667,0.1833
9,0,cf_9,,,,7,,,18.9745,7,,0.8756,3,True,0.1667,0.0567


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.89,,,,,
9,0,cf_9,,,,7,,,18.9745,7,,0.8756,3,True,0.1667,0.0567
1,1,xin,3,4,1,2,3,0,22.3757,0,62.56,,,,,
10,1,cf_1,,,4,,,,22.3757,4,,0.8796,3,True,0.52,0.1967
11,1,cf_2,,,,3,1,,22.3757,,,0.8796,3,True,0.52,0.02
12,1,cf_4,,,,,1,,22.3757,1,,0.8796,3,True,0.52,0.1233
13,1,cf_5,2,,,,1,,22.3757,,,0.8796,3,True,0.52,0.0367
14,1,cf_6,,,,7,,,22.3757,7,,0.8796,3,True,0.52,0.1867
15,1,cf_8,,1,,7,,,22.3757,7,,0.8796,4,True,0.52,0.23
16,1,cf_9,1,2,,,,,22.3757,2,,0.8796,4,True,0.52,0.0867


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,2.89,,,,,
9,0,cf_9,,,,7,,,18.9745,7,,0.8756,3,True,0.1667,0.0567
1,1,xin,3,4,1,2,3,0,22.3757,0,62.56,,,,,
10,1,cf_1,,,4,,,,22.3757,4,,0.8796,3,True,0.52,0.1967
2,2,xin,5,3,1,1,4,0,22.694,7,3.04,,,,,
11,2,cf_9,,,2,6,1,,22.68,,,0.8753,4,True,0.3333,0.07
3,3,xin,3,3,6,6,2,0,24.3418,1,3.29,,,,,
12,3,,,,,,,,,,,,,False,,
4,4,xin,5,4,2,7,1,0,21.2585,3,3.39,,,,,
13,4,cf_2,,,6,,,,21.2585,,,0.9868,2,True,0.0067,0.0


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_9 ---
Predicted risk: 0.0567
Valid: True
Changes:
  alcfreq: 4 → 7
  bmi: 18.9866 → 18.9745
  dosprt: 0 → 7


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.1967
Valid: True
Changes:
  cgtsmok: 1 → 4
  dosprt: 0 → 4

--- cf_2 ---
Predicted risk: 0.02
Valid: True
Changes:
  alcfreq: 2 → 3
  slprl: 3 → 1

--- cf_4 ---
Predicted risk: 0.1233
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1

--- cf_5 ---
Predicted risk: 0.0367
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1

--- cf_6 ---
Predicted risk: 0.1867
Valid: True
Changes:
  alcfreq: 2 → 7
  dosprt